# 04 — Model Training & Evaluation
All 5 models trained and evaluated using:
1. **Train/test split** — direct evaluation on held-out test set
2. **LOOCV** — Leave-One-Out CV on full dataset (robust for small n=25)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle, os, warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from xgboost import XGBRegressor

os.makedirs('results/metrics',  exist_ok=True)
os.makedirs('results/figures',  exist_ok=True)

with open('data/processed/arrays.pkl', 'rb') as f:
    d = pickle.load(f)

X_train, y_train = d['X_train'], d['y_train']
X_test,  y_test  = d['X_test'],  d['y_test']
X_all,   y_all   = d['X_all'],   d['y_all']

FEATURES = ['gate','and','inv','nor','nand','or','dff','in','out']
COLORS   = {'SVR':'#4C72B0','KNN':'#DD8452','RF':'#55A868',
            'XGBoost':'#C44E52','BPNN':'#8172B2'}
print("✓ Data loaded")


In [ ]:
# ── Define all 5 models ──
MODELS = {
    'SVR':     SVR(kernel='rbf', C=100, epsilon=0.01),
    'KNN':     KNeighborsRegressor(n_neighbors=3),
    'RF':      RandomForestRegressor(n_estimators=200, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.05,
                             random_state=42, verbosity=0),
    'BPNN':    MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=2000,
                             activation='relu', random_state=42)
}
print("Models defined:", list(MODELS.keys()))


In [ ]:
# ── Train/Test Evaluation ──
print("="*55)
print("  TRAIN/TEST SPLIT RESULTS")
print("="*55)

split_rows  = []
trained     = {}
predictions = {}

for name, model in MODELS.items():
    model.fit(X_train, y_train)
    trained[name]     = model
    preds             = model.predict(X_test)
    predictions[name] = preds
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)
    mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
    split_rows.append({'Model':name,'RMSE':rmse,'MAE':mae,'R²':r2,'MAPE(%)':mape})
    print(f"  {name:10s}  RMSE={rmse:.5f}  MAE={mae:.5f}  R²={r2:.4f}  MAPE={mape:.2f}%")

split_df = pd.DataFrame(split_rows)
split_df.to_csv('results/metrics/split_metrics.csv', index=False)

# Save trained models
with open('data/processed/trained_models.pkl', 'wb') as f:
    pickle.dump(trained, f)
print("\n✓ Trained models saved")


In [ ]:
# ── LOOCV Evaluation ──
print("="*55)
print("  LOOCV RESULTS (n=25, robust estimate)")
print("="*55)

loocv_rows = []
loo        = LeaveOneOut()

for name, model in MODELS.items():
    preds = cross_val_predict(model, X_all, y_all, cv=loo)
    rmse  = np.sqrt(mean_squared_error(y_all, preds))
    mae   = mean_absolute_error(y_all, preds)
    r2    = r2_score(y_all, preds)
    loocv_rows.append({'Model':name,'RMSE_LOOCV':rmse,'MAE_LOOCV':mae,'R²_LOOCV':r2})
    print(f"  {name:10s}  RMSE={rmse:.5f}  MAE={mae:.5f}  R²={r2:.4f}")

loocv_df = pd.DataFrame(loocv_rows)
loocv_df.to_csv('results/metrics/loocv_metrics.csv', index=False)
print("\n✓ LOOCV metrics saved")


In [ ]:
# ── Figure 1: Metric comparison bar charts ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Model Comparison — ISCAS\'89 Power Prediction", fontsize=13)

for ax, metric in zip(axes, ['RMSE', 'MAE', 'R²']):
    vals   = split_df[metric]
    colors = [COLORS[m] for m in split_df['Model']]
    bars   = ax.bar(split_df['Model'], vals, color=colors,
                    edgecolor='white', linewidth=0.8)
    ax.set_title(metric, fontsize=11)
    ax.set_ylabel(metric)
    ylim = vals.max() * 1.3 if metric != 'R²' else 1.05
    ax.set_ylim(0, ylim)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + vals.max()*0.03,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/figures/fig1_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Figure 1 saved")


In [ ]:
# ── Figure 2: Predicted vs Actual ──
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
fig.suptitle("Predicted vs Actual Power — Test Set", fontsize=12)

for ax, name in zip(axes, MODELS.keys()):
    preds = predictions[name]
    ax.scatter(y_test, preds, color=COLORS[name], s=80,
               edgecolors='white', linewidth=0.6, zorder=3)
    mn = min(y_test.min(), preds.min()) * 0.9
    mx = max(y_test.max(), preds.max()) * 1.1
    ax.plot([mn, mx], [mn, mx], 'k--', linewidth=1, alpha=0.5)
    r2 = r2_score(y_test, preds)
    ax.text(0.05, 0.90, f'R²={r2:.3f}', transform=ax.transAxes,
            fontsize=8, bbox=dict(boxstyle='round,pad=0.3',
            facecolor='white', alpha=0.8))
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Actual (W)', fontsize=8)
    ax.set_ylabel('Predicted (W)', fontsize=8)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/figures/fig2_predicted_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Figure 2 saved")


In [ ]:
# ── Figure 3: LOOCV vs Split RMSE side-by-side ──
merged = split_df[['Model','RMSE']].merge(
         loocv_df[['Model','RMSE_LOOCV']], on='Model')

x     = np.arange(len(merged))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width/2, merged['RMSE'],       width, label='Train/Test split',
       color=[COLORS[m] for m in merged['Model']], alpha=0.9, edgecolor='white')
ax.bar(x + width/2, merged['RMSE_LOOCV'], width, label='LOOCV',
       color=[COLORS[m] for m in merged['Model']], alpha=0.45, edgecolor='white',
       hatch='//')
ax.set_xticks(x)
ax.set_xticklabels(merged['Model'])
ax.set_ylabel('RMSE')
ax.set_title('RMSE: Train/Test Split vs LOOCV')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/figures/fig3_split_vs_loocv.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Figure 3 saved")


---
**Paper note:** Report LOOCV metrics as the primary result (more reliable for n=25).  
Mention train/test split results in ablation/discussion section.

**Next:** Run `05_shap_analysis.ipynb` ← **your novelty**